In [6]:
# import os
# import glob
# import yaml
# import json
# import pandas as pd
# from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# def parse_pipeline(base_dir="logs"):
#     results = []
    
#     # 1. 遍历所有的日期文件夹和时间文件夹
#     run_dirs = glob.glob(os.path.join(base_dir, "*", "*/"))
#     print(f"🔍 找到 {len(run_dirs)} 个实验文件夹，正在启动优先级级联解析...")

#     for run_dir in run_dirs:
#         try:
#             # 各种配置文件的路径
#             lightning_dir = os.path.join(run_dir, "lightning_logs", "version_0")
#             hparams_path = os.path.join(lightning_dir, "hparams.yaml")
#             config_path = os.path.join(run_dir, "hydra_configs", "config.yaml")
#             overrides_path = os.path.join(run_dir, "hydra_configs", "overrides.yaml")
            
#             # 第一步：检查 TensorBoard 和基础配置是否存在
#             tb_files = glob.glob(os.path.join(lightning_dir, "events.out.tfevents.*"))
#             if not tb_files or not os.path.exists(hparams_path) or not os.path.exists(config_path):
#                 continue

#             # 第二步：加载所有 YAML 文件建立字典
#             # A. 基础 Config (兜底)
#             with open(config_path, 'r', encoding='utf-8') as f:
#                 config_dict = yaml.safe_load(f) or {}
                
#             # B. 初始超参 (次高优先级)
#             with open(hparams_path, 'r', encoding='utf-8') as f:
#                 hparams_dict = yaml.safe_load(f) or {}
                
#             # C. 手动 Overrides (最高优先级)
#             overrides_dict = {}
#             if os.path.exists(overrides_path):
#                 with open(overrides_path, 'r', encoding='utf-8') as f:
#                     ov_list = yaml.safe_load(f) or []
#                     for item in ov_list:
#                         if "=" in item:
#                             k, v = item.split("=", 1)
#                             overrides_dict[k.strip()] = v.strip()

#             # --- 🚀 核心：设计一个级联读取器 (Overrides > HParams > Config) ---
#             def get_hp(ov_key, hp_key=None, default="N/A"):
#                 if ov_key in overrides_dict:
#                     return overrides_dict[ov_key]
#                 if hp_key and hp_key in hparams_dict:
#                     return hparams_dict[hp_key]
#                 return default

#             # 第三步：提取和清洗关键信息
#             # 1. 模型名称提取 (去前缀)
#             model_target = get_hp("module._target_", None, config_dict.get("module", {}).get("_target_", "Unknown"))
#             model_name = overrides_dict.get("model", str(model_target).split(".")[-1]) # 如果覆盖了 model=rnn_ctc 则优先用缩写

#             # 2. 数据增强方法清洗 (把 ${to_tensor} 变成 to_tensor)
#             transforms_raw = config_dict.get("transforms", {}).get("train", [])
#             transforms_clean = [str(t).replace("${", "").replace("}", "") for t in transforms_raw]
#             transforms_str = " + ".join(transforms_clean) if transforms_clean else "None"

#             # 3. 超参提取
#             layers = get_hp("module.num_layers", "num_layers")
#             hidden = get_hp("module.hidden_size", "hidden_size")
#             dropout = get_hp("module.dropout", "dropout")
#             bidirectional = get_hp("module.bidirectional", "bidirectional")
#             act = get_hp("module.nonlinearity", "nonlinearity", "N/A")
#             hop_len = get_hp("logspec.hop_length", None, config_dict.get("logspec", {}).get("hop_length", "N/A"))

#             # 第四步：解析 TensorBoard 的真实表现
#             ea = EventAccumulator(lightning_dir)
#             ea.Reload()
#             tags = ea.Tags()['scalars']
            
#             # 检查 Test 指标和 Epoch > 5
#             required_test_metrics = ["test/CER", "test/DER", "test/IER", "test/SER", "test/loss"]
#             if not all(metric in tags for metric in required_test_metrics):
#                 continue
                
#             if "epoch" not in tags:
#                 continue
#             max_epoch = max([e.value for e in ea.Scalars("epoch")])
#             if max_epoch <= 5:
#                 continue
                
#             def get_final_metric(metric_name):
#                 return ea.Scalars(metric_name)[-1].value if metric_name in tags else None

#             # 组装黄金数据字典
#             run_name = run_dir.strip(os.sep).split(os.sep)[-2:] 
#             results.append({
#                 "Run_ID": f"{run_name[0]}/{run_name[1]}",
#                 "Epochs": int(max_epoch),
                
#                 # 级联解析出来的全套配置
#                 "Model": model_name,
#                 "Transforms": transforms_str,
#                 "Layers": layers,
#                 "Hidden": hidden,
#                 "Dropout": dropout,
#                 "Bidir": bidirectional,
#                 "Act": act,
#                 "Hop_Len": hop_len,
                
#                 # 考试成绩
#                 "Test CER": get_final_metric("test/CER"),
#                 "Test DER": get_final_metric("test/DER"),
#                 "Test IER": get_final_metric("test/IER"),
#                 "Test SER": get_final_metric("test/SER"),
#                 "Test Loss": get_final_metric("test/loss"),
#                 "Val CER": get_final_metric("val/CER")
#             })

#         except Exception as e:
#             pass # 静默跳过损坏的文件夹

#     # 导出和展示
#     if results:
#         valid_results = sorted(results, key=lambda x: x["Test CER"])
        
#         with open("final_tensorboard_results.json", "w", encoding="utf-8") as f:
#             json.dump(valid_results, f, indent=4)
            
#         df = pd.DataFrame(valid_results)
#         df.to_csv("final_tensorboard_results.csv", index=False)
        
#         print("\n" + "="*90)
#         print(f"✅ 大功告成！基于优先级级联解析，成功提炼出 {len(valid_results)} 组黄金实验记录！")
#         print("🥇 目前 Test CER 排名前三的模型是：")
#         print(df[["Run_ID", "Model", "Transforms", "Layers", "Hidden", "Test CER"]].head(3).to_string(index=False))
#         print("="*90)
#     else:
#         print("\n🤷‍♂️ 没有找到同时满足【跑完 Test】且【Epoch > 5】的完整实验。")

# if __name__ == "__main__":
#     parse_pipeline()


import os
import glob
import yaml
import json
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def parse_pipeline(base_dir="logs"):
    results = []
    
    run_dirs = glob.glob(os.path.join(base_dir, "*", "*/"))
    print(f"🔍 找到 {len(run_dirs)} 个实验文件夹，正在启动带 Window Length 的终极解析...")

    for run_dir in run_dirs:
        try:
            lightning_dir = os.path.join(run_dir, "lightning_logs", "version_0")
            hparams_path = os.path.join(lightning_dir, "hparams.yaml")
            config_path = os.path.join(run_dir, "hydra_configs", "config.yaml")
            overrides_path = os.path.join(run_dir, "hydra_configs", "overrides.yaml")
            
            tb_files = glob.glob(os.path.join(lightning_dir, "events.out.tfevents.*"))
            if not tb_files or not os.path.exists(hparams_path) or not os.path.exists(config_path):
                continue

            with open(config_path, 'r', encoding='utf-8') as f:
                config_dict = yaml.safe_load(f) or {}
                
            with open(hparams_path, 'r', encoding='utf-8') as f:
                hparams_dict = yaml.safe_load(f) or {}
                
            overrides_dict = {}
            if os.path.exists(overrides_path):
                with open(overrides_path, 'r', encoding='utf-8') as f:
                    ov_list = yaml.safe_load(f) or []
                    for item in ov_list:
                        if "=" in item:
                            k, v = item.split("=", 1)
                            overrides_dict[k.strip()] = v.strip()

            def get_hp(ov_key, hp_key=None, default="N/A"):
                if ov_key in overrides_dict:
                    return overrides_dict[ov_key]
                if hp_key and hp_key in hparams_dict:
                    return hparams_dict[hp_key]
                return default

            model_target = get_hp("module._target_", None, config_dict.get("module", {}).get("_target_", "Unknown"))
            model_name = overrides_dict.get("model", str(model_target).split(".")[-1])

            transforms_raw = config_dict.get("transforms", {}).get("train", [])
            transforms_clean = [str(t).replace("${", "").replace("}", "") for t in transforms_raw]
            transforms_str = " + ".join(transforms_clean) if transforms_clean else "None"

            # 提取所有超参数
            layers = get_hp("module.num_layers", "num_layers")
            hidden = get_hp("module.hidden_size", "hidden_size")
            dropout = get_hp("module.dropout", "dropout")
            bidirectional = get_hp("module.bidirectional", "bidirectional")
            act = get_hp("module.nonlinearity", "nonlinearity", "N/A")
            hop_len = get_hp("logspec.hop_length", None, config_dict.get("logspec", {}).get("hop_length", "N/A"))
            
            # 🎯 核心新增：提取 Window Length
            # 先去 overrides 里找 datamodule.window_length，找不到就去 config.yaml 里的 datamodule 组找
            window_len = get_hp("datamodule.window_length", None, config_dict.get("datamodule", {}).get("window_length", "N/A"))

            ea = EventAccumulator(lightning_dir)
            ea.Reload()
            tags = ea.Tags()['scalars']
            
            required_test_metrics = ["test/CER", "test/DER", "test/IER", "test/SER", "test/loss"]
            if not all(metric in tags for metric in required_test_metrics):
                continue
                
            if "epoch" not in tags:
                continue
            max_epoch = max([e.value for e in ea.Scalars("epoch")])
            if max_epoch <= 5:
                continue
                
            def get_final_metric(metric_name):
                return ea.Scalars(metric_name)[-1].value if metric_name in tags else None

            results.append({
                "Run_ID": f"{run_dir.strip(os.sep).split(os.sep)[-2]}/{run_dir.strip(os.sep).split(os.sep)[-1]}",
                "Epochs": int(max_epoch),
                
                "Model": model_name,
                "Window_Len": window_len,  # <--- 加进字典里！
                "Hop_Len": hop_len,
                "Transforms": transforms_str,
                "Layers": layers,
                "Hidden": hidden,
                "Dropout": dropout,
                "Bidir": bidirectional,
                "Act": act,
                
                "Test CER": get_final_metric("test/CER"),
                "Test DER": get_final_metric("test/DER"),
                "Test IER": get_final_metric("test/IER"),
                "Test SER": get_final_metric("test/SER"),
                "Test Loss": get_final_metric("test/loss"),
                "Val CER": get_final_metric("val/CER")
            })

        except Exception as e:
            pass

    if results:
        valid_results = sorted(results, key=lambda x: x["Test CER"])
        
        with open("final_tensorboard_results.json", "w", encoding="utf-8") as f:
            json.dump(valid_results, f, indent=4)
            
        df = pd.DataFrame(valid_results)
        df.to_csv("final_tensorboard_results.csv", index=False)
        
        print("\n" + "="*100)
        print(f"✅ 成功！提取出了 {len(valid_results)} 组包含 Window Length 的完整实验记录！")
        print("🥇 目前 Test CER 排名前三的模型是：")
        # 把 Window_Len 加到打印的列里
        print(df[["Run_ID", "Model", "Window_Len", "Hop_Len", "Layers", "Hidden", "Test CER"]].head(3).to_string(index=False))
        print("="*100)
    else:
        print("\n🤷‍♂️ 没有找到完整实验。")

if __name__ == "__main__":
    parse_pipeline()

🔍 找到 191 个实验文件夹，正在启动带 Window Length 的终极解析...

✅ 成功！提取出了 136 组包含 Window Length 的完整实验记录！
🥇 目前 Test CER 排名前三的模型是：
             Run_ID    Model Window_Len Hop_Len Layers Hidden  Test CER
2026-03-06/02-49-24 lstm_ctc       8000      16      2    512 15.517614
2026-03-06/03-37-06 lstm_ctc       8000      16      2    512 17.851740
2026-03-06/02-49-20 lstm_ctc       8000      16      2    256 18.348822


In [3]:


import os
import glob
import yaml
import json
import re
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def parse_pipeline(base_dir="logs"):
    results = []
    
    run_dirs = glob.glob(os.path.join(base_dir, "*", "*/"))
    run_dirs=r"L:\workspace\logs"
    print(f"🔍 找到 {len(run_dirs)} 个实验文件夹，正在启动带 Transformer 特化参数的终极解析...")

    for run_dir in run_dirs:
        try:
            lightning_dir = os.path.join(run_dir, "lightning_logs", "version_0")
            hparams_path = os.path.join(lightning_dir, "hparams.yaml")
            config_path = os.path.join(run_dir, "hydra_configs", "config.yaml")
            overrides_path = os.path.join(run_dir, "hydra_configs", "overrides.yaml")
            ckpt_dir = os.path.join(run_dir, "checkpoints")
            
            tb_files = glob.glob(os.path.join(lightning_dir, "events.out.tfevents.*"))
            if not tb_files or not os.path.exists(hparams_path) or not os.path.exists(config_path):
                continue

            with open(config_path, 'r', encoding='utf-8') as f:
                config_dict = yaml.safe_load(f) or {}
                
            with open(hparams_path, 'r', encoding='utf-8') as f:
                hparams_dict = yaml.safe_load(f) or {}
                
            overrides_dict = {}
            if os.path.exists(overrides_path):
                with open(overrides_path, 'r', encoding='utf-8') as f:
                    ov_list = yaml.safe_load(f) or []
                    for item in ov_list:
                        if "=" in item:
                            k, v = item.split("=", 1)
                            overrides_dict[k.strip()] = v.strip()

            def get_hp(ov_key, hp_key=None, default="N/A"):
                if ov_key in overrides_dict:
                    return overrides_dict[ov_key]
                if hp_key and hp_key in hparams_dict:
                    return hparams_dict[hp_key]
                return default

            model_target = get_hp("module._target_", None, config_dict.get("module", {}).get("_target_", "Unknown"))
            model_name = overrides_dict.get("model", str(model_target).split(".")[-1])

            transforms_raw = config_dict.get("transforms", {}).get("train", [])
            transforms_clean = [str(t).replace("${", "").replace("}", "") for t in transforms_raw]
            transforms_str = " + ".join(transforms_clean) if transforms_clean else "None"

            # === 提取所有通用超参数 ===
            layers = get_hp("module.num_layers", "num_layers")
            hidden = get_hp("module.hidden_size", "hidden_size")
            dropout = get_hp("module.dropout", "dropout")
            bidirectional = get_hp("module.bidirectional", "bidirectional")
            act = get_hp("module.nonlinearity", "nonlinearity", "N/A")
            hop_len = get_hp("logspec.hop_length", None, config_dict.get("logspec", {}).get("hop_length", "N/A"))
            window_len = get_hp("datamodule.window_length", None, config_dict.get("datamodule", {}).get("window_length", "N/A"))

            # 🎯 核心新增：提取 Transformer 专属参数
            nhead = get_hp("module.nhead", "nhead", "N/A")
            dim_ff = get_hp("module.dim_feedforward", "dim_feedforward", "N/A")

            # 提取 Best Epoch
            best_epoch = "N/A"
            if os.path.exists(ckpt_dir):
                for ckpt_file in os.listdir(ckpt_dir):
                    match = re.search(r'epoch=(\d+)', ckpt_file)
                    if match and "last.ckpt" not in ckpt_file:
                        best_epoch = int(match.group(1))
                        break 

            ea = EventAccumulator(lightning_dir)
            ea.Reload()
            tags = ea.Tags()['scalars']
            
            required_test_metrics = ["test/CER", "test/DER", "test/IER", "test/SER", "test/loss"]
            if not all(metric in tags for metric in required_test_metrics):
                continue
                
            if "epoch" not in tags:
                continue
            max_epoch = max([e.value for e in ea.Scalars("epoch")])
            if max_epoch <= 2: # 放宽限制，以防有只跑了几个epoch就停掉的实验
                continue
                
            def get_final_metric(metric_name):
                return ea.Scalars(metric_name)[-1].value if metric_name in tags else None

            results.append({
                "Run_ID": f"{run_dir.strip(os.sep).split(os.sep)[-2]}/{run_dir.strip(os.sep).split(os.sep)[-1]}",
                "Total_Epochs": int(max_epoch),
                "Best_Epoch": best_epoch,
                
                "Model": model_name,
                "Window_Len": window_len, 
                "Hop_Len": hop_len,
                "Transforms": transforms_str,
                "Layers": layers,
                
                # 🎯 添加 Transformer 列，保留旧的 Hidden 列
                "NHead": nhead,
                "Dim_FF": dim_ff,
                "Hidden": hidden,
                
                "Dropout": dropout,
                "Bidir": bidirectional,
                "Act": act,
                
                "Test CER": get_final_metric("test/CER"),
                "Test DER": get_final_metric("test/DER"),
                "Test IER": get_final_metric("test/IER"),
                "Test SER": get_final_metric("test/SER"),
                "Test Loss": get_final_metric("test/loss"),
                "Val CER": get_final_metric("val/CER")
            })

        except Exception as e:
            pass

    if results:
        valid_results = sorted(results, key=lambda x: x["Test CER"])
        
        with open("final_tensorboard_results.json", "w", encoding="utf-8") as f:
            json.dump(valid_results, f, indent=4)
            
        df = pd.DataFrame(valid_results)
        df.to_csv("final_tensorboard_results_Ctrans.csv", index=False)
        
        print("\n" + "="*120)
        print(f"✅ 成功！提取出了 {len(valid_results)} 组完整实验记录（包含 Transformer 参数）！")
        print("🥇 目前 Test CER 排名前三的模型是：")
        
        # 打印时加上 NHead 和 Dim_FF
        print(df[["Run_ID", "Model", "Window_Len", "Hop_Len", "Layers", "NHead", "Dim_FF", "Hidden", "Test CER"]].head(3).to_string(index=False))
        print("="*120)
    else:
        print("\n🤷‍♂️ 没有找到完整实验。")

if __name__ == "__main__":
    parse_pipeline()

🔍 找到 17 个实验文件夹，正在启动带 Transformer 特化参数的终极解析...

🤷‍♂️ 没有找到完整实验。
